# Cargamos el dataset con feature_eng

In [26]:
pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [27]:
import sys
print(sys.executable)

c:\miniforge3\envs\dp261-g2\python.exe


In [28]:
# ========================
# 1. Cargar dataset
# ========================

import pandas as pd

df = pd.read_csv('../data/interim/hotel_bookings_fe.csv')

print("Shape:", df.shape)
df.head()

Shape: (86372, 51)


,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,adults,children,babies,country,...,total_guests,price_per_person,is_loyal,arrival_month_num,is_high_season,lead_time_group,price_per_person_total,lead_time_x_nights,adr_log,lead_time_log
0,0,7,2015,July,1,0,1,0,0,GBR,...,1,75.0,0,7,1,last_minute,75.0,7,4.330733,2.079442
1,0,13,2015,July,1,0,1,0,0,GBR,...,1,75.0,0,7,1,short,75.0,13,4.330733,2.639057
2,0,14,2015,July,1,0,2,0,0,GBR,...,2,49.0,0,7,1,short,98.0,28,4.595120,2.708050
3,0,0,2015,July,1,0,2,0,0,PRT,...,2,53.5,0,7,1,NaN,107.0,0,4.682131,0.000000
4,0,9,2015,July,1,0,2,0,0,PRT,...,2,51.5,0,7,1,short,103.0,18,4.644391,2.302585


# Análisis del target

In [29]:
# ========================
# 2. Distribución del target
# ========================

target_dist = df['is_canceled'].value_counts(normalize=True)

print(target_dist)

is_canceled
0    0.725293
1    0.274707
Name: proportion, dtype: float64


# Dividimos el dataset

In [30]:
# ========================
# Separar variables
# ========================

X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

In [31]:
# ========================
# 3. Train/Test Split
# ========================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (69097, 50)
Test: (17275, 50)


In [32]:
# ========================
# 4. Distribución antes de balancear
# ========================

print("Antes de SMOTE:")
print(y_train.value_counts())

Antes de SMOTE:
is_canceled
0    50116
1    18981
Name: count, dtype: int64


# Aplicamos método de balanceo

In [34]:
X_train.select_dtypes(include='object').columns

C:\Users\Luz Maria\AppData\Local\Temp\ipykernel_16028\1697935278.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train.select_dtypes(include='object').columns


Index(['arrival_date_month', 'country', 'lead_time_group'], dtype='str')

In [35]:
# ========================
# Eliminar variables categóricas no numéricas
# ========================

cols_to_drop = ['arrival_date_month', 'country', 'lead_time_group']

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print("Columnas eliminadas:", cols_to_drop)

Columnas eliminadas: ['arrival_date_month', 'country', 'lead_time_group']


In [36]:
# ========================
# 5. Aplicar SMOTE
# ========================

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

Se identificaron variables categóricas en formato texto que no podían ser procesadas por SMOTE. Estas variables fueron eliminadas, ya que sus versiones transformadas o numéricas ya estaban disponibles en el dataset.

In [37]:
# ========================
# 5. Aplicar SMOTE
# ========================

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

In [38]:
print(y_train.value_counts())
print(y_train_bal.value_counts())

is_canceled
0    50116
1    18981
Name: count, dtype: int64
is_canceled
1    50116
0    50116
Name: count, dtype: int64


# Guardamos la base procesada

In [39]:
# ========================
# Guardar TRAIN balanceado
# ========================

X_train_bal.to_csv('../data/processed/X_train_bal.csv', index=False)
y_train_bal.to_csv('../data/processed/y_train_bal.csv', index=False)

In [40]:
# ========================
# Guardar TEST (sin modificar)
# ========================

X_test.to_csv('../data/processed/X_test.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)